In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "src"))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\mayan\PROJECTS\SIPAR


In [2]:
import geemap
import pandas as pd

from kerala_flood.config import load_config
from kerala_flood.gee_auth import initialize_ee
from kerala_flood.aoi import load_tile_roi
from kerala_flood.sentinel1 import load_sentinel1_collection, make_median_composite
from kerala_flood.flood_detection import (
    smooth_sar,
    make_permanent_water_mask,
    make_terrain_mask,
    detect_flood_by_change,
    calculate_flood_area_sqkm
)
from kerala_flood.export import export_image_to_drive

In [3]:
config = load_config()

config

{'project_id': 'sipar-498412',
 'tile_id': 'Tile_054',
 'paths': {'tile_geojson': 'data/boundaries/kerala_boundary_tiles.geojson'},
 'sentinel1': {'polarization': 'VV',
  'orbit_pass': 'DESCENDING',
  'instrument_mode': 'IW'},
 'dates': {'pre_start': '2018-07-20',
  'pre_end': '2018-08-01',
  'flood_start': '2018-08-15',
  'flood_end': '2018-08-31'},
 'thresholds': {'change_db': -2.5,
  'permanent_water_occurrence': 90,
  'slope_threshold': 15},
 'export': {'scale': 10, 'folder': 'Kerala_Flood_Tile024_MVP'}}

In [4]:
initialize_ee(config["project_id"])

Earth Engine initialized


In [5]:
tile_path = PROJECT_ROOT / config["paths"]["tile_geojson"]

roi = load_tile_roi(
    tile_geojson_path=str(tile_path),
    tile_id=config["tile_id"]
)

print("ROI loaded:", config["tile_id"])

ROI loaded: Tile_054


In [6]:
print("Selected tile:", config["tile_id"])
print("Tile GeoJSON:", tile_path)

Selected tile: Tile_054
Tile GeoJSON: C:\Users\mayan\PROJECTS\SIPAR\data\boundaries\kerala_boundary_tiles.geojson


In [7]:
m = geemap.Map()
m.centerObject(roi, 10)
m.addLayer(roi, {"color": "red"}, config["tile_id"])
m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [18]:
print("Selected tile:", config["tile_id"])
print("Tile GeoJSON path:", tile_path)
print("Project root:", PROJECT_ROOT)

Selected tile: Tile_054
Tile GeoJSON path: C:\Users\mayan\PROJECTS\SIPAR\data\boundaries\kerala_boundary_tiles.geojson
Project root: C:\Users\mayan\PROJECTS\SIPAR


In [19]:
s1 = load_sentinel1_collection(
    roi=roi,
    polarization=config["sentinel1"]["polarization"],
    orbit_pass=config["sentinel1"]["orbit_pass"],
    instrument_mode=config["sentinel1"]["instrument_mode"]
)

print("Sentinel-1 collection created.")
print("Polarization:", config["sentinel1"]["polarization"])
print("Orbit pass:", config["sentinel1"]["orbit_pass"])

Sentinel-1 collection created.
Polarization: VV
Orbit pass: DESCENDING


In [20]:
pre_img, pre_count = make_median_composite(
    collection=s1,
    start_date=config["dates"]["pre_start"],
    end_date=config["dates"]["pre_end"],
    roi=roi
)

flood_img, flood_count = make_median_composite(
    collection=s1,
    start_date=config["dates"]["flood_start"],
    end_date=config["dates"]["flood_end"],
    roi=roi
)

pre_count_value = pre_count.getInfo()
flood_count_value = flood_count.getInfo()

print("Pre-flood image count:", pre_count_value)
print("Flood-period image count:", flood_count_value)

Pre-flood image count: 3
Flood-period image count: 4


In [22]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    pre_img,
    {"min": -25, "max": 0},
    "Pre-flood VV"
)

m.addLayer(
    flood_img,
    {"min": -25, "max": 0},
    "Flood-period VV"
)

m.addLayer(
    roi,
    {"color": "red"},
    f"ROI {config['tile_id']}"
)

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [23]:
pre_smooth = smooth_sar(pre_img, radius_m=30)
flood_smooth = smooth_sar(flood_img, radius_m=30)

print("SAR smoothing completed.")

SAR smoothing completed.


In [24]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    pre_smooth,
    {"min": -25, "max": 0},
    "Pre-flood VV smoothed"
)

m.addLayer(
    flood_smooth,
    {"min": -25, "max": 0},
    "Flood-period VV smoothed"
)

m.addLayer(
    roi,
    {"color": "red"},
    "ROI"
)

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [25]:
permanent_water = make_permanent_water_mask(
    roi=roi,
    occurrence_threshold=config["thresholds"]["permanent_water_occurrence"]
)

print("Permanent water mask created.")
print("Occurrence threshold:", config["thresholds"]["permanent_water_occurrence"])

Permanent water mask created.
Occurrence threshold: 90


In [26]:
terrain_mask, slope = make_terrain_mask(
    roi=roi,
    slope_threshold=config["thresholds"]["slope_threshold"]
)

print("Terrain mask created.")
print("Slope threshold:", config["thresholds"]["slope_threshold"])

Terrain mask created.
Slope threshold: 15


In [27]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    permanent_water.selfMask(),
    {"palette": ["blue"]},
    "Permanent water"
)

m.addLayer(
    slope,
    {"min": 0, "max": 30, "palette": ["green", "yellow", "red"]},
    "Slope"
)

m.addLayer(
    roi,
    {"color": "black"},
    "ROI"
)

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [28]:
change_db, flood_mask, confidence = detect_flood_by_change(
    pre_image=pre_smooth,
    flood_image=flood_smooth,
    permanent_water=permanent_water,
    terrain_mask=terrain_mask,
    change_threshold_db=config["thresholds"]["change_db"]
)

print("Flood detection completed.")
print("Change threshold dB:", config["thresholds"]["change_db"])

Flood detection completed.
Change threshold dB: -2.5


In [29]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    change_db,
    {
        "min": -8,
        "max": 8,
        "palette": ["red", "white", "blue"]
    },
    "Change dB"
)

m.addLayer(
    flood_mask,
    {"palette": ["cyan"]},
    "Flood mask"
)

m.addLayer(
    confidence,
    {
        "min": 0,
        "max": 1,
        "palette": ["red", "yellow", "green"]
    },
    "Confidence"
)

m.addLayer(
    roi,
    {"color": "black"},
    "ROI"
)

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [12]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    change_db,
    {"min": -8, "max": 8, "palette": ["red", "white", "blue"]},
    "Change dB"
)

m.addLayer(
    permanent_water.selfMask(),
    {"palette": ["blue"]},
    "Permanent Water"
)

m.addLayer(
    flood_mask,
    {"palette": ["cyan"]},
    "Flood Mask"
)

m.addLayer(
    confidence,
    {"min": 0, "max": 1, "palette": ["red", "yellow", "green"]},
    "Confidence"
)

m.addLayer(roi, {"color": "black"}, "ROI")

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [30]:
flood_area_sqkm = calculate_flood_area_sqkm(
    flood_mask=flood_mask,
    roi=roi,
    scale=config["export"]["scale"]
)

print(f"Flood area for {config['tile_id']}: {flood_area_sqkm:.2f} sq km")

Flood area for Tile_054: 1.46 sq km


In [31]:
summary = pd.DataFrame([{
    "tile_id": config["tile_id"],
    "pre_start": config["dates"]["pre_start"],
    "pre_end": config["dates"]["pre_end"],
    "flood_start": config["dates"]["flood_start"],
    "flood_end": config["dates"]["flood_end"],
    "polarization": config["sentinel1"]["polarization"],
    "orbit_pass": config["sentinel1"]["orbit_pass"],
    "pre_image_count": pre_count_value,
    "flood_image_count": flood_count_value,
    "change_threshold_db": config["thresholds"]["change_db"],
    "permanent_water_occurrence": config["thresholds"]["permanent_water_occurrence"],
    "slope_threshold": config["thresholds"]["slope_threshold"],
    "flood_area_sqkm": flood_area_sqkm
}])

output_table = PROJECT_ROOT / "outputs" / "tables" / "tile024_mvp_summary.csv"
output_table.parent.mkdir(parents=True, exist_ok=True)

summary.to_csv(output_table, index=False)

print("Saved summary to:", output_table)

summary

Saved summary to: C:\Users\mayan\PROJECTS\SIPAR\outputs\tables\tile024_mvp_summary.csv


,tile_id,pre_start,pre_end,flood_start,flood_end,polarization,orbit_pass,pre_image_count,flood_image_count,change_threshold_db,permanent_water_occurrence,slope_threshold,flood_area_sqkm
0,Tile_054,2018-07-20,2018-08-01,2018-08-15,2018-08-31,VV,DESCENDING,3,4,-2.5,90,15,1.461091


In [32]:
export_image_to_drive(
    image=flood_mask.unmask(0).toByte(),
    description="Tile024_MVP_Flood_Mask",
    folder=config["export"]["folder"],
    file_name="tile024_mvp_flood_mask",
    region=roi,
    scale=config["export"]["scale"]
)

export_image_to_drive(
    image=change_db,
    description="Tile024_MVP_Change_dB",
    folder=config["export"]["folder"],
    file_name="tile024_mvp_change_db",
    region=roi,
    scale=config["export"]["scale"]
)

export_image_to_drive(
    image=confidence.unmask(0),
    description="Tile024_MVP_Confidence",
    folder=config["export"]["folder"],
    file_name="tile024_mvp_confidence",
    region=roi,
    scale=config["export"]["scale"]
)

Started export: Tile024_MVP_Flood_Mask
Started export: Tile024_MVP_Change_dB
Started export: Tile024_MVP_Confidence


<Task 63XVEBDGLUNMIKE476GBDYMN Type.EXPORT_IMAGE: Tile024_MVP_Confidence (State.UNSUBMITTED)>

In [34]:

import ee
tile_area_sqkm = (
    ee.Image.pixelArea()
    .reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=roi,
        scale=config["export"]["scale"],
        maxPixels=1e13
    )
    .getInfo()
)

tile_area_value = list(tile_area_sqkm.values())[0] / 1e6

flood_percent = (flood_area_sqkm / tile_area_value) * 100

print(f"Tile area: {tile_area_value:.2f} sq km")
print(f"Flood area: {flood_area_sqkm:.2f} sq km")
print(f"Flood percentage: {flood_percent:.2f}%")

Tile area: 755.12 sq km
Flood area: 1.46 sq km
Flood percentage: 0.19%


In [35]:
thresholds = [-1.5, -2.0, -2.5, -3.0, -4.0]

rows = []

for threshold in thresholds:
    change_tmp, flood_tmp, confidence_tmp = detect_flood_by_change(
        pre_image=pre_smooth,
        flood_image=flood_smooth,
        permanent_water=permanent_water,
        terrain_mask=terrain_mask,
        change_threshold_db=threshold
    )

    area_tmp = calculate_flood_area_sqkm(
        flood_mask=flood_tmp,
        roi=roi,
        scale=config["export"]["scale"]
    )

    rows.append({
        "threshold_db": threshold,
        "flood_area_sqkm": area_tmp
    })

threshold_df = pd.DataFrame(rows)
threshold_df

,threshold_db,flood_area_sqkm
0,-1.5,3.198066
1,-2.0,2.171898
2,-2.5,1.461091
3,-3.0,0.963051
4,-4.0,0.442107


In [37]:
mask_settings = {
    "sensitive": -2.0,
    "balanced": -2.5,
    "conservative": -3.0
}

final_masks = {}
final_areas = []

for mask_name, threshold in mask_settings.items():
    change_tmp, flood_tmp, confidence_tmp = detect_flood_by_change(
        pre_image=pre_smooth,
        flood_image=flood_smooth,
        permanent_water=permanent_water,
        terrain_mask=terrain_mask,
        change_threshold_db=threshold
    )

    area_tmp = calculate_flood_area_sqkm(
        flood_mask=flood_tmp,
        roi=roi,
        scale=config["export"]["scale"]
    )

    final_masks[mask_name] = {
        "threshold": threshold,
        "change_db": change_tmp,
        "flood_mask": flood_tmp,
        "confidence": confidence_tmp,
        "area_sqkm": area_tmp
    }

    final_areas.append({
        "mask_type": mask_name,
        "threshold_db": threshold,
        "flood_area_sqkm": area_tmp
    })

final_area_df = pd.DataFrame(final_areas)
final_area_df

,mask_type,threshold_db,flood_area_sqkm
0,sensitive,-2.0,2.171898
1,balanced,-2.5,1.461091
2,conservative,-3.0,0.963051


In [38]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    change_db,
    {"min": -8, "max": 8, "palette": ["red", "white", "blue"]},
    "Change dB"
)

m.addLayer(
    final_masks["sensitive"]["flood_mask"],
    {"palette": ["yellow"]},
    "Sensitive flood mask (-2.0 dB)",
    shown=False
)

m.addLayer(
    final_masks["balanced"]["flood_mask"],
    {"palette": ["cyan"]},
    "Balanced flood mask (-2.5 dB)",
    shown=True
)

m.addLayer(
    final_masks["conservative"]["flood_mask"],
    {"palette": ["blue"]},
    "Conservative flood mask (-3.0 dB)",
    shown=False
)

m.addLayer(roi, {"color": "black"}, "ROI")

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [39]:
output_table = PROJECT_ROOT / "outputs" / "tables" / "tile024_threshold_sensitivity.csv"
output_table.parent.mkdir(parents=True, exist_ok=True)

threshold_df.to_csv(output_table, index=False)

print("Saved:", output_table)

output_table = PROJECT_ROOT / "outputs" / "tables" / "tile024_final_mask_areas.csv"

final_area_df.to_csv(output_table, index=False)

print("Saved:", output_table)

Saved: C:\Users\mayan\PROJECTS\SIPAR\outputs\tables\tile024_threshold_sensitivity.csv
Saved: C:\Users\mayan\PROJECTS\SIPAR\outputs\tables\tile024_final_mask_areas.csv


In [40]:
mask_settings = {
    "sensitive": -2.0,
    "balanced": -2.5,
    "conservative": -3.0
}

final_masks = {}
final_areas = []

for mask_name, threshold in mask_settings.items():
    change_tmp, flood_tmp, confidence_tmp = detect_flood_by_change(
        pre_image=pre_smooth,
        flood_image=flood_smooth,
        permanent_water=permanent_water,
        terrain_mask=terrain_mask,
        change_threshold_db=threshold
    )

    area_tmp = calculate_flood_area_sqkm(
        flood_mask=flood_tmp,
        roi=roi,
        scale=config["export"]["scale"]
    )

    final_masks[mask_name] = {
        "threshold": threshold,
        "change_db": change_tmp,
        "flood_mask": flood_tmp,
        "confidence": confidence_tmp,
        "area_sqkm": area_tmp
    }

    final_areas.append({
        "tile_id": config["tile_id"],
        "mask_type": mask_name,
        "threshold_db": threshold,
        "flood_area_sqkm": area_tmp
    })

final_area_df = pd.DataFrame(final_areas)
final_area_df

,tile_id,mask_type,threshold_db,flood_area_sqkm
0,Tile_054,sensitive,-2.0,2.171898
1,Tile_054,balanced,-2.5,1.461091
2,Tile_054,conservative,-3.0,0.963051


In [41]:
output_table = PROJECT_ROOT / "outputs" / "tables" / "tile024_final_mask_areas.csv"
output_table.parent.mkdir(parents=True, exist_ok=True)

final_area_df.to_csv(output_table, index=False)

print("Saved:", output_table)
final_area_df

Saved: C:\Users\mayan\PROJECTS\SIPAR\outputs\tables\tile024_final_mask_areas.csv


,tile_id,mask_type,threshold_db,flood_area_sqkm
0,Tile_054,sensitive,-2.0,2.171898
1,Tile_054,balanced,-2.5,1.461091
2,Tile_054,conservative,-3.0,0.963051


In [42]:
m = geemap.Map()
m.centerObject(roi, 10)

m.addLayer(
    change_db,
    {"min": -8, "max": 8, "palette": ["red", "white", "blue"]},
    "Change dB"
)

m.addLayer(
    final_masks["sensitive"]["flood_mask"],
    {"palette": ["yellow"]},
    "Sensitive mask -2.0 dB",
    shown=False
)

m.addLayer(
    final_masks["balanced"]["flood_mask"],
    {"palette": ["cyan"]},
    "Balanced mask -2.5 dB",
    shown=True
)

m.addLayer(
    final_masks["conservative"]["flood_mask"],
    {"palette": ["blue"]},
    "Conservative mask -3.0 dB",
    shown=False
)

m.addLayer(roi, {"color": "black"}, "ROI")

m

Map(center=[11.172339841268903, 75.99110836784095], controls=(WidgetControl(options=['position', 'transparent_…

In [43]:
export_image_to_drive(
    image=final_masks["sensitive"]["flood_mask"].unmask(0).toByte(),
    description="Tile024_Sensitive_Flood_Mask",
    folder=config["export"]["folder"],
    file_name="tile024_sensitive_flood_mask",
    region=roi,
    scale=config["export"]["scale"]
)

export_image_to_drive(
    image=final_masks["balanced"]["flood_mask"].unmask(0).toByte(),
    description="Tile024_Balanced_Flood_Mask",
    folder=config["export"]["folder"],
    file_name="tile024_balanced_flood_mask",
    region=roi,
    scale=config["export"]["scale"]
)

export_image_to_drive(
    image=final_masks["conservative"]["flood_mask"].unmask(0).toByte(),
    description="Tile024_Conservative_Flood_Mask",
    folder=config["export"]["folder"],
    file_name="tile024_conservative_flood_mask",
    region=roi,
    scale=config["export"]["scale"]
)

export_image_to_drive(
    image=change_db,
    description="Tile024_Change_dB",
    folder=config["export"]["folder"],
    file_name="tile024_change_db",
    region=roi,
    scale=config["export"]["scale"]
)

Started export: Tile024_Sensitive_Flood_Mask
Started export: Tile024_Balanced_Flood_Mask
Started export: Tile024_Conservative_Flood_Mask
Started export: Tile024_Change_dB


<Task 7MSQOXRQ2NKI3UANX2EG2Z7U Type.EXPORT_IMAGE: Tile024_Change_dB (State.UNSUBMITTED)>